### Step 1 — Configure Python path

Purpose: Add `src/` to `sys.path` so `ibnr_utils` is importable without installation, handling both `notebooks/` and project-root working directories.  
Uses: `pathlib.Path`, `sys.path`.  
Produces: `src_path` resolved and appended to `sys.path`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

src_path = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

### Step 2 — Import ibnr_utils functions

Purpose: Import all functions used in this notebook.  
Uses: `ata_matrix`, `atu_matrix`, `avg_ldfs`, `calculate_ratio`, `common_actual_ratios`, `create_triangles_from_project_database`, `ldf`, `latest_cumulative_actual`, `ratio_frame`, `ultimate_matrix`.  
Produces: Module symbols available in the session.

In [2]:
from ibnr_utils import (
    ata_matrix,
    atu_matrix,
    avg_ldfs,
    calculate_ratio,
    common_actual_ratios,
    create_triangles_from_project_database,
    ldf,
    latest_cumulative_actual,
    ratio_frame,
    ultimate_matrix,
)

### Step 3 — Load triangles

Purpose: Load the Loss Incurred triangle for Development Method work and load all concepts for ratio layout construction.  
Uses: `create_triangles_from_project_database`.  
Produces: `loss_incurred` — Loss Incurred monthly triangle; `triangles` — full concept dict (all concepts, silent validation).

In [3]:
result = create_triangles_from_project_database(
    basis="month",
    concepts="Loss Incurred",
)

triangles = create_triangles_from_project_database(
    basis="month",
    print_validation=False,
).triangles

loss_incurred = result.triangles["Loss Incurred"]
loss_incurred.shape

All validations passed.
All validations passed.


(120, 120)

### Step 4 — Inspect the Loss Incurred triangle

Purpose: Display the full Loss Incurred triangle to verify shape and `NaN` pattern.  
Produces: Loss Incurred triangle DataFrame.

In [4]:
loss_incurred

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,...,dev_110,dev_111,dev_112,dev_113,dev_114,dev_115,dev_116,dev_117,dev_118,dev_119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,153700.711642,168265.054672,205478.444379,184336.168001,201248.543480,196453.059025,176881.337833,200214.346199,187578.698914,190075.991440,...,177018.978866,171085.713030,173585.908595,169160.747063,173081.527124,176445.361233,172720.085632,173112.466318,174926.788130,175796.196292
2016-02,174526.085249,179397.620865,205041.059805,195258.548978,187048.160649,201436.435289,199293.923502,205210.386921,242546.688042,200341.751962,...,183520.470365,177750.670575,178954.148728,181846.011293,186153.778295,183854.533150,183737.684382,180857.272493,183746.248505,NaN
2016-03,139977.733603,194087.931093,215736.346315,175783.066907,217515.571324,183537.102418,202607.299157,199878.253026,206739.049289,236638.405603,...,185889.013660,185870.257200,187689.323174,188042.155009,189089.878262,185853.845022,186063.868425,184100.501488,NaN,NaN
2016-04,179441.145220,171328.983083,197378.848951,208702.402696,208493.696364,223233.698165,187690.613593,253992.426802,225375.117705,243903.465956,...,200265.915955,199299.153969,197234.175592,195849.823081,193656.903165,201506.058677,196951.796202,NaN,NaN,NaN
2016-05,185553.143033,216977.705207,209628.471276,214214.723923,232801.660391,210101.993900,201936.310336,223423.445555,230653.011303,211583.631794,...,209927.848694,207676.902404,209507.332802,209279.782933,205822.853384,201165.375549,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,362141.012374,432661.332612,445636.521558,541592.036871,378209.883552,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,367005.573515,415446.207539,433412.965713,518520.051557,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,345630.285842,346963.701868,393892.055949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 5 — Compute the LDF (link-ratio) triangle

Purpose: Derive the age-to-age link-ratio triangle from the cumulative triangle. Division errors are filled with 1.0; empty cells remain `NaN`.  
Uses: `ldf`.  
Produces: `loss_incurred_ldf` — DataFrame of link ratios with same index/columns as the source triangle; shape confirmation displayed.

In [5]:
loss_incurred_ldf = ldf(loss_incurred)
loss_incurred_ldf.shape

(119, 119)

### Step 6 — Display the full LDF triangle

Purpose: Inspect the complete link-ratio triangle before computing averages.  
Produces: `loss_incurred_ldf` displayed in full.

In [6]:
loss_incurred_ldf

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,...,dev_110,dev_111,dev_112,dev_113,dev_114,dev_115,dev_116,dev_117,dev_118,dev_119
accident_period,,,,,,,,,,,,,,,,,,,,,
2016-01,1.094758,1.221159,0.897107,1.091747,0.976171,0.900375,1.131913,0.936889,1.013313,1.049322,...,0.996083,0.966482,1.014614,0.974507,1.023178,1.019435,0.978887,1.002272,1.010481,1.00497
2016-02,1.027913,1.142942,0.952290,0.957951,1.076923,0.989364,1.029687,1.181942,0.825993,0.994134,...,0.988735,0.968560,1.006771,1.016160,1.023689,0.987649,0.999364,0.984323,1.015974,NaN
2016-03,1.386563,1.111539,0.814805,1.237409,0.843788,1.103904,0.986530,1.034325,1.144624,0.955552,...,1.011841,0.999899,1.009787,1.001880,1.005572,0.982886,1.001130,0.989448,NaN,NaN
2016-04,0.954792,1.152046,1.057370,0.999000,1.070698,0.840781,1.353251,0.887330,1.082211,0.948795,...,1.008606,0.995173,0.989639,0.992981,0.988803,1.040531,0.977399,NaN,NaN,NaN
2016-05,1.169356,0.966129,1.021878,1.086768,0.902494,0.961135,1.106406,1.032358,0.917324,1.014616,...,1.000932,0.989278,1.008814,0.998914,0.983482,0.977371,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07,0.717988,1.323178,0.982921,1.041007,1.041904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08,1.194732,1.029989,1.215322,0.698330,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,1.131989,1.043247,1.196365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 7 — Display an LDF slice

Purpose: Examine the first 10 accident periods × 10 development periods for a compact view of early-development link ratios.  
Produces: 10×10 slice of `loss_incurred_ldf`.

In [7]:
loss_incurred_ldf.iloc[:10, :10]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10
accident_period,,,,,,,,,,
2016-01,1.094758,1.221159,0.897107,1.091747,0.976171,0.900375,1.131913,0.936889,1.013313,1.049322
2016-02,1.027913,1.142942,0.952290,0.957951,1.076923,0.989364,1.029687,1.181942,0.825993,0.994134
2016-03,1.386563,1.111539,0.814805,1.237409,0.843788,1.103904,0.986530,1.034325,1.144624,0.955552
2016-04,0.954792,1.152046,1.057370,0.999000,1.070698,0.840781,1.353251,0.887330,1.082211,0.948795
2016-05,1.169356,0.966129,1.021878,1.086768,0.902494,0.961135,1.106406,1.032358,0.917324,1.014616
2016-06,1.000480,0.974894,0.894022,1.015167,0.903941,1.383779,0.860752,1.100394,0.912101,0.859247
2016-07,1.220314,0.869438,1.000130,1.084071,0.899339,0.869194,1.103414,1.091474,1.250649,1.054984
2016-08,1.305943,0.952623,0.952412,1.105079,1.015613,1.034524,0.880366,0.976919,1.086770,1.038386
2016-09,0.912501,1.029687,0.863905,1.057685,1.214017,0.840839,1.021144,1.138338,1.055682,1.013752


## Average LDFs

### Step 8 — Simple average LDF, latest 12 periods

Purpose: Compute unweighted arithmetic mean of the 12 most-recent link ratios for each development column.  
Uses: `avg_ldfs(average_type="simple", latest_periods=12)`.  
Produces: `simple_latest_12` — one-row average Series per development column; first 12 columns displayed.

In [8]:
simple_latest_12 = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple",
    latest_periods=12,
)

simple_latest_12.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S - 12,0.950224,1.084801,1.071894,0.976335,1.011227,1.044487,0.984723,1.108333,0.971907,1.014538,0.980418,1.055502


### Step 9 — Simple average excluding min/max, latest 12 periods

Purpose: Compute unweighted mean of latest 12 link ratios after removing the single highest and lowest value per column.  
Uses: `avg_ldfs(average_type="simple_excluding_min_max", latest_periods=12)`.  
Produces: `simple_latest_12_excluding_min_max`.

In [9]:
simple_latest_12_excluding_min_max = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple_excluding_min_max",
    latest_periods=12,
)

simple_latest_12_excluding_min_max.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S Excl Min/Max - 12,0.948997,1.07843,1.06725,0.985332,0.998466,1.037621,0.974483,1.095034,0.954675,1.024911,0.975198,1.048597


### Step 10 — Simple average excluding ±2 std, latest 12 periods

Purpose: Compute unweighted mean of latest 12 link ratios after excluding values outside ±2 standard deviations from the 12-period mean.  
Uses: `avg_ldfs(average_type="simple_excluding_std", latest_periods=12, std_deviations=2)`.  
Produces: `simple_latest_12_within_2_std`.

In [10]:
simple_latest_12_within_2_std = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple_excluding_std",
    latest_periods=12,
    std_deviations=2,
)

simple_latest_12_within_2_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S Excl 2 Std - 12 - Std 12,0.950224,1.063131,1.071894,1.001608,0.97583,1.044487,0.9608,1.064519,0.935374,1.039483,0.980418,1.055502


### Step 11 — Simple average excluding ±1 std, latest 12 periods

Purpose: Compute unweighted mean of latest 12 link ratios after excluding values outside ±1 standard deviation computed from the same 12-period window.  
Uses: `avg_ldfs(average_type="simple_excluding_std", latest_periods=12, std_deviations=1)`.  
Produces: `simple_latest_12_within_1_std`.

In [11]:
simple_latest_12_within_1_std = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple_excluding_std",
    latest_periods=12,
    std_deviations=1,
)

simple_latest_12_within_1_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S Excl 1 Std - 12 - Std 12,0.942164,1.091219,1.048136,0.950683,0.992245,1.030679,0.974483,1.118803,0.918575,1.009997,0.954893,1.030902


### Step 12 — Simple average ±1 std with 24-period standard-deviation band, latest 12

Purpose: Compute unweighted mean of latest 12 link ratios, using a wider 24-period window to estimate the standard-deviation band before excluding outliers.  
Uses: `avg_ldfs(average_type="simple_excluding_std", latest_periods=12, std_periods=24, std_deviations=1)`.  
Produces: `simple_latest_12_within_1_std_using_latest_24_std`.  

Interpretation: Using `std_periods > latest_periods` stabilises the exclusion boundary when recent data is sparse or volatile without expanding the averaging window itself.

In [12]:
simple_latest_12_within_1_std_using_latest_24_std = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple_excluding_std",
    latest_periods=12,
    std_periods=24,
    std_deviations=1,
)

simple_latest_12_within_1_std_using_latest_24_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S Excl 1 Std - 12 - Std 24,0.975917,1.051207,1.051719,1.001608,0.992245,1.050287,0.963944,1.075505,1.009437,0.995253,0.954893,1.048597


### Step 13 — Manual verification: 1-std inclusion window for dev_1 (12-period band)

Purpose: Show the raw link-ratio values and inclusion flags for `dev_1` under the 12-period standard-deviation band, to verify that `avg_ldfs` excludes the expected observations.  
Produces: DataFrame of `dev_1` factors with Boolean `include_within_1_std` column.

In [13]:
latest_12_dev_1 = loss_incurred_ldf["dev_1"].dropna().tail(12)
dev_1_mean = latest_12_dev_1.mean()
dev_1_std = latest_12_dev_1.std(ddof=1)
dev_1_lower_bound = dev_1_mean - dev_1_std
dev_1_upper_bound = dev_1_mean + dev_1_std

pd.DataFrame(
    {
        "dev_1_factor": latest_12_dev_1,
        "include_within_1_std": latest_12_dev_1.between(
            dev_1_lower_bound,
            dev_1_upper_bound,
        ),
    }
)

,dev_1_factor,include_within_1_std
accident_period,,
2024-12,0.927016,True
2025-01,0.861964,True
2025-02,0.978166,True
2025-03,0.962939,True
2025-04,0.939591,True
2025-05,0.984869,True
2025-06,0.878912,True
2025-07,0.717988,False
2025-08,1.194732,False


### Step 14 — Manual verification: 1-std inclusion window for dev_1 (24-period band)

Purpose: Show the same `dev_1` values with inclusion flags computed from the wider 24-period standard-deviation band, to confirm the difference in boundary relative to Step 13.  
Produces: DataFrame of `dev_1` factors with Boolean `include_using_latest_24_std` column.

In [14]:
latest_24_dev_1_for_std = loss_incurred_ldf["dev_1"].dropna().tail(24)
dev_1_mean_24 = latest_24_dev_1_for_std.mean()
dev_1_std_24 = latest_24_dev_1_for_std.std(ddof=1)
dev_1_lower_bound_24 = dev_1_mean_24 - dev_1_std_24
dev_1_upper_bound_24 = dev_1_mean_24 + dev_1_std_24

pd.DataFrame(
    {
        "dev_1_factor": latest_12_dev_1,
        "include_using_latest_24_std": latest_12_dev_1.between(
            dev_1_lower_bound_24,
            dev_1_upper_bound_24,
        ),
    }
)

,dev_1_factor,include_using_latest_24_std
accident_period,,
2024-12,0.927016,True
2025-01,0.861964,False
2025-02,0.978166,True
2025-03,0.962939,True
2025-04,0.939591,True
2025-05,0.984869,True
2025-06,0.878912,True
2025-07,0.717988,False
2025-08,1.194732,False


### Step 15 — Simple average with selected cell exclusions, latest 12

Purpose: Compute unweighted mean of latest 12 link ratios, manually removing specific (accident period, development column) cells before averaging.  
Uses: `avg_ldfs(average_type="simple", latest_periods=12, exclude_cells={"dev_1": ["2025-07", "2025-11"]})`.  
Produces: `simple_latest_12_with_selected_exclusions`.

In [15]:
simple_latest_12_with_selected_exclusions = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple",
    latest_periods=12,
    exclude_cells={"dev_1": ["2025-07", "2025-11"]},
)

simple_latest_12_with_selected_exclusions.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S - 12 - Sel Excl,0.983978,1.084801,1.071894,0.976335,1.011227,1.044487,0.984723,1.108333,0.971907,1.014538,0.980418,1.055502


### Step 16 — Simple average ±1 std (24-period band) with selected exclusions, latest 12

Purpose: Combine std-deviation exclusion (24-period band) with manual cell exclusions to demonstrate that both exclusion types can be applied simultaneously.  
Uses: `avg_ldfs(average_type="simple_excluding_std", latest_periods=12, std_periods=24, std_deviations=1, exclude_cells={...})`.  
Produces: `simple_latest_12_with_selected_exclusions_and_latest_24_std`.

In [16]:
simple_latest_12_with_selected_exclusions_and_latest_24_std = avg_ldfs(
    loss_incurred_ldf,
    average_type="simple_excluding_std",
    latest_periods=12,
    std_periods=24,
    std_deviations=1,
    exclude_cells={"dev_1": ["2025-07", "2025-11"]},
)

simple_latest_12_with_selected_exclusions_and_latest_24_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S Excl 1 Std - 12 - Std 24 - Sel Excl,1.004931,1.051207,1.051719,1.001608,0.992245,1.050287,0.963944,1.075505,1.009437,0.995253,0.954893,1.048597


## Volume-Weighted Average LDFs

### Step 17 — Volume-weighted average LDF, latest 12 periods

Purpose: Compute age-to-age link ratios weighted by the prior-age cumulative triangle values, using the 12 most-recent periods.  
Uses: `avg_ldfs(cumulative_triangle=loss_incurred, average_type="volume_weighted", latest_periods=12)`.  
Produces: `volume_weighted_latest_12`.

In [17]:
volume_weighted_latest_12 = avg_ldfs(
    loss_incurred_ldf,
    cumulative_triangle=loss_incurred,
    average_type="volume_weighted",
    latest_periods=12,
)

volume_weighted_latest_12.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
VW - 12,0.939165,1.078846,1.073926,0.963653,0.998379,1.028142,0.981623,1.092813,0.956561,1.000826,0.978253,1.05308


### Step 18 — Volume-weighted average excluding min/max, latest 12

Purpose: Volume-weighted mean of latest 12 link ratios after removing the single highest and lowest value per column.  
Uses: `avg_ldfs(average_type="volume_weighted_excluding_min_max", latest_periods=12)`.  
Produces: `volume_weighted_latest_12_excluding_min_max`.

In [18]:
volume_weighted_latest_12_excluding_min_max = avg_ldfs(
    loss_incurred_ldf,
    cumulative_triangle=loss_incurred,
    average_type="volume_weighted_excluding_min_max",
    latest_periods=12,
)

volume_weighted_latest_12_excluding_min_max.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
VW Excl Min/Max - 12,0.945259,1.073633,1.070606,0.981612,0.993429,1.025248,0.973988,1.09117,0.948216,1.021603,0.97425,1.049573


### Step 19 — Volume-weighted average ±1 std (24-period band), latest 12

Purpose: Volume-weighted mean of latest 12 link ratios after excluding values outside ±1 standard deviation, computed from a 24-period window.  
Uses: `avg_ldfs(average_type="volume_weighted_excluding_std", latest_periods=12, std_periods=24, std_deviations=1)`.  
Produces: `volume_weighted_latest_12_within_1_std_using_latest_24_std`.

In [19]:
volume_weighted_latest_12_within_1_std_using_latest_24_std = avg_ldfs(
    loss_incurred_ldf,
    cumulative_triangle=loss_incurred,
    average_type="volume_weighted_excluding_std",
    latest_periods=12,
    std_periods=24,
    std_deviations=1,
)

volume_weighted_latest_12_within_1_std_using_latest_24_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
VW Excl 1 Std - 12 - Std 24,0.970381,1.046216,1.055137,0.995377,0.989535,1.05054,0.96313,1.071537,1.005232,0.99311,0.953808,1.049573


### Step 20 — Volume-weighted average ±1 std (24-period band) with selected exclusions

Purpose: Combine volume-weighted std-deviation exclusion with manual cell exclusions.  
Uses: `avg_ldfs(average_type="volume_weighted_excluding_std", latest_periods=12, std_periods=24, std_deviations=1, exclude_cells={...})`.  
Produces: `volume_weighted_latest_12_with_selected_exclusions_and_latest_24_std`.

In [20]:
volume_weighted_latest_12_with_selected_exclusions_and_latest_24_std = avg_ldfs(
    loss_incurred_ldf,
    cumulative_triangle=loss_incurred,
    average_type="volume_weighted_excluding_std",
    latest_periods=12,
    std_periods=24,
    std_deviations=1,
    exclude_cells={"dev_1": ["2025-07", "2025-11"]},
)

volume_weighted_latest_12_with_selected_exclusions_and_latest_24_std.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
VW Excl 1 Std - 12 - Std 24 - Sel Excl,1.004714,1.046216,1.055137,0.995377,0.989535,1.05054,0.96313,1.071537,1.005232,0.99311,0.953808,1.049573


### Step 21 — Inspect selected factor count by column

Purpose: Retrieve the number of link ratios that survived all exclusion filters for each development column, stored as a metadata attribute on the result.  
Produces: `selected_factor_count_by_column` dict — column name → count of included factors.

In [21]:
simple_latest_12_with_selected_exclusions.attrs["selected_factor_count_by_column"]

{'dev_1': 12,
 'dev_2': 12,
 'dev_3': 12,
 'dev_4': 12,
 'dev_5': 12,
 'dev_6': 12,
 'dev_7': 12,
 'dev_8': 12,
 'dev_9': 12,
 'dev_10': 12,
 'dev_11': 12,
 'dev_12': 12,
 'dev_13': 12,
 'dev_14': 12,
 'dev_15': 12,
 'dev_16': 12,
 'dev_17': 12,
 'dev_18': 12,
 'dev_19': 12,
 'dev_20': 12,
 'dev_21': 12,
 'dev_22': 12,
 'dev_23': 12,
 'dev_24': 12,
 'dev_25': 12,
 'dev_26': 12,
 'dev_27': 12,
 'dev_28': 12,
 'dev_29': 12,
 'dev_30': 12,
 'dev_31': 12,
 'dev_32': 12,
 'dev_33': 12,
 'dev_34': 12,
 'dev_35': 12,
 'dev_36': 12,
 'dev_37': 12,
 'dev_38': 12,
 'dev_39': 12,
 'dev_40': 12,
 'dev_41': 12,
 'dev_42': 12,
 'dev_43': 12,
 'dev_44': 12,
 'dev_45': 12,
 'dev_46': 12,
 'dev_47': 12,
 'dev_48': 12,
 'dev_49': 12,
 'dev_50': 12,
 'dev_51': 12,
 'dev_52': 12,
 'dev_53': 12,
 'dev_54': 12,
 'dev_55': 12,
 'dev_56': 12,
 'dev_57': 12,
 'dev_58': 12,
 'dev_59': 12,
 'dev_60': 12,
 'dev_61': 12,
 'dev_62': 12,
 'dev_63': 12,
 'dev_64': 12,
 'dev_65': 12,
 'dev_66': 12,
 'dev_67': 12,
 'de

## Average LDF Comparison

### Step 22 — Compare all average LDF methods

Purpose: Stack all computed average rows into a single DataFrame for side-by-side comparison across average types and exclusion strategies.  
Uses: `pd.concat`.  
Produces: `average_comparison` — multi-row DataFrame; first 12 columns displayed.

In [22]:
average_comparison = pd.concat(
    [
        simple_latest_12,
        simple_latest_12_excluding_min_max,
        simple_latest_12_within_2_std,
        simple_latest_12_within_1_std,
        simple_latest_12_within_1_std_using_latest_24_std,
        simple_latest_12_with_selected_exclusions,
        simple_latest_12_with_selected_exclusions_and_latest_24_std,
        volume_weighted_latest_12,
        volume_weighted_latest_12_excluding_min_max,
        volume_weighted_latest_12_within_1_std_using_latest_24_std,
        volume_weighted_latest_12_with_selected_exclusions_and_latest_24_std,
    ]
)

average_comparison.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
average_type,,,,,,,,,,,,
S - 12,0.950224,1.084801,1.071894,0.976335,1.011227,1.044487,0.984723,1.108333,0.971907,1.014538,0.980418,1.055502
S Excl Min/Max - 12,0.948997,1.078430,1.067250,0.985332,0.998466,1.037621,0.974483,1.095034,0.954675,1.024911,0.975198,1.048597
S Excl 2 Std - 12 - Std 12,0.950224,1.063131,1.071894,1.001608,0.975830,1.044487,0.960800,1.064519,0.935374,1.039483,0.980418,1.055502
S Excl 1 Std - 12 - Std 12,0.942164,1.091219,1.048136,0.950683,0.992245,1.030679,0.974483,1.118803,0.918575,1.009997,0.954893,1.030902
S Excl 1 Std - 12 - Std 24,0.975917,1.051207,1.051719,1.001608,0.992245,1.050287,0.963944,1.075505,1.009437,0.995253,0.954893,1.048597
S - 12 - Sel Excl,0.983978,1.084801,1.071894,0.976335,1.011227,1.044487,0.984723,1.108333,0.971907,1.014538,0.980418,1.055502
S Excl 1 Std - 12 - Std 24 - Sel Excl,1.004931,1.051207,1.051719,1.001608,0.992245,1.050287,0.963944,1.075505,1.009437,0.995253,0.954893,1.048597
VW - 12,0.939165,1.078846,1.073926,0.963653,0.998379,1.028142,0.981623,1.092813,0.956561,1.000826,0.978253,1.053080
VW Excl Min/Max - 12,0.945259,1.073633,1.070606,0.981612,0.993429,1.025248,0.973988,1.091170,0.948216,1.021603,0.974250,1.049573


## ATA Matrix

### Step 23 — Define ATA average specifications

Purpose: Declare the ordered list of named average methods that will populate the ATA matrix rows.  
Produces: `average_specs` — list of dicts specifying method, latest_periods, and optional exclusion flags for each named row.

In [23]:
average_specs = [
    {"method": "simple", "latest_periods": 12, "label": "S - 12"},
    {
        "method": "simple",
        "latest_periods": 12,
        "exclude_min_max": True,
        "label": "S Excl Min/Max - 12",
    },
    {
        "method": "simple",
        "latest_periods": 12,
        "exclude_std": True,
        "std_deviations": 1,
        "std_periods": 24,
        "label": "S Excl 1 Std - 12 - Std 24",
    },
    {"method": "weighted", "latest_periods": 12, "label": "VW - 12"},
    {
        "method": "weighted",
        "latest_periods": 24,
        "exclude_min_max": True,
        "label": "VW Excl Min/Max - 24",
    },
]

### Step 24 — Build the ATA matrix with selection

Purpose: Construct the age-to-age (ATA) matrix containing all average rows, a Prior Selected row, and a Selected row assembled from block-range rules, column overrides, and a forced-to-one tail region.  
Uses: `ata_matrix(loss_incurred, average_specs, selected=..., prior_selected=..., selection_ranges=..., selection_by_column=..., force_one_after=..., tail_factor=...)`.  
Produces: `ata` — ATA matrix DataFrame; key columns displayed.  

Interpretation: `selection_ranges` applies one average method block-uniformly across a column range. `selection_by_column` overrides individual columns. `force_one_after` sets all selected factors beyond a specified column to 1.0 (no further development assumed). `tail_factor=1.000001` appends a near-unity tail without materially altering projections.

In [24]:
ata = ata_matrix(
    loss_incurred,
    average_specs,
    selected="S Excl Min/Max - 12",
    prior_selected={
        "dev_1": 1.00,
        "dev_2": 1.00,
        "dev_3": 1.00,
        "tail": 1.00,
    },
    selection_ranges=[
        {"row": "S - 12", "start": "dev_1", "end": "dev_12"},
        {"row": "VW - 12", "start": "dev_13", "end": "dev_24"},
    ],
    selection_by_column={
        "dev_35": "S Excl 1 Std - 12 - Std 24",
        "dev_45": "VW Excl Min/Max - 24",
    },
    force_one_after="dev_48",
    tail_factor=1.000001,
)

ata.loc[:, ["dev_1", "dev_12", "dev_13", "dev_24", "dev_35", "dev_45", "dev_48", "dev_49", "tail"]]

,dev_1,dev_12,dev_13,dev_24,dev_35,dev_45,dev_48,dev_49,tail
average_type,,,,,,,,,
S - 12,0.950224,1.055502,1.011987,0.988935,1.008995,1.025775,0.978876,0.995885,1.000001
S Excl Min/Max - 12,0.948997,1.048597,1.002140,0.987428,1.011838,1.032178,0.982383,0.990679,1.000001
S Excl 1 Std - 12 - Std 24,0.975917,1.048597,0.946376,1.003012,1.023665,1.025782,0.990575,0.990479,1.000001
VW - 12,0.939165,1.053080,0.997685,0.988595,1.009314,1.025371,0.977552,0.995716,1.000001
VW Excl Min/Max - 24,1.005732,1.065044,0.968379,0.994248,1.012751,1.019923,0.984605,0.996441,1.000001
Prior Selected,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
Selected,0.950224,1.055502,0.997685,0.988595,1.023665,1.019923,0.982383,1.000000,1.000001


## ATU And Percent Developed

### Step 25 — Build the ATU and percent-developed matrix

Purpose: Compute cumulative age-to-ultimate (ATU) factors and the corresponding percent-developed vector by taking running products of the Selected row in the ATA matrix.  
Uses: `atu_matrix(ata)`.  
Produces: `atu` — DataFrame containing ATU factors and `% Developed` per development column; first 12 columns displayed.  

Interpretation: ATU is never floored. Selected LDFs below 1.0 produce ATU values below 1.0 and `% Developed` above 1.0; diagnostic flags in `ultimate_matrix` surface these conditions without changing the formula.

In [25]:
atu = atu_matrix(ata)
atu.iloc[:, :12]

,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,dev_10,dev_11,dev_12
Selected ATA,0.950224,1.084801,1.071894,0.976335,1.011227,1.044487,0.984723,1.108333,0.971907,1.014538,0.980418,1.055502
Prior Selected ATA,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
ATU,1.160389,1.221174,1.125712,1.050208,1.075664,1.063722,1.018416,1.034215,0.933127,0.960099,0.946341,0.965242
Prior ATU,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
% Developed,0.861780,0.818884,0.888326,0.952192,0.929658,0.940095,0.981917,0.966917,1.071666,1.041559,1.056702,1.036009
Prior % Developed,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


**Note:** Selected LDFs below 1.0 produce ATU values below 1.0 and Ultimate values below Actual for the affected accident periods. This is the transparent result of applying the selected factors — ATU and Ultimate are not floored. Review the ATU row and the `% Developed` values exceeding 1.0 before accepting ultimates that fall below the latest diagonal.

## Ultimate Matrix

### Step 26 — Build the ultimate matrix

Purpose: Apply the next applicable ATU factor to each accident period's latest observed cumulative diagonal to produce projected ultimates.  
Uses: `ultimate_matrix(loss_incurred, atu)`.  
Produces: `ultimate` — DataFrame with columns `Latest`, `ATU`, `% Developed`, `Ultimate`, and diagnostic flag columns.

In [26]:
ultimate = ultimate_matrix(loss_incurred, atu)
ultimate

,actual_development_period,atu_period,Actual,ATU,% Developed,Ultimate,atu_below_1,percent_developed_above_1,ultimate_below_actual
accident_period,,,,,,,,,
2016-01,dev_119,tail,175796.196292,1.000001,0.999999,175796.372088,False,False,False
2016-02,dev_118,dev_119,183746.248505,1.000001,0.999999,183746.432251,False,False,False
2016-03,dev_117,dev_118,184100.501488,1.000001,0.999999,184100.685588,False,False,False
2016-04,dev_116,dev_117,196951.796202,1.000001,0.999999,196951.993154,False,False,False
2016-05,dev_115,dev_116,201165.375549,1.000001,0.999999,201165.576714,False,False,False
...,...,...,...,...,...,...,...,...,...
2025-08,dev_4,dev_5,378209.883552,1.075664,0.929658,406826.914791,False,False,False
2025-09,dev_3,dev_4,518520.051557,1.050208,0.952192,544554.142622,False,False,False
2025-10,dev_2,dev_3,393892.055949,1.125712,0.888326,443409.111759,False,False,False


### Step 27 — Inspect the most-recent accident periods

Purpose: Display the last 12 rows of the ultimate matrix to review projected ultimates for the most-immature accident periods where development uncertainty is highest.  
Produces: Last 12 rows of `ultimate`.

In [27]:
ultimate.tail(12)

,actual_development_period,atu_period,Actual,ATU,% Developed,Ultimate,atu_below_1,percent_developed_above_1,ultimate_below_actual
accident_period,,,,,,,,,
2025-01,dev_11,dev_12,371810.967683,0.965242,1.036009,358887.733562,True,True,True
2025-02,dev_10,dev_11,389069.574465,0.946341,1.056702,368192.460875,True,True,True
2025-03,dev_9,dev_10,378665.391843,0.960099,1.041559,363556.238918,True,True,True
2025-04,dev_8,dev_9,391032.636077,0.933127,1.071666,364883.076350,True,True,True
2025-05,dev_7,dev_8,469677.406931,1.034215,0.966917,485747.406309,False,False,False
2025-06,dev_6,dev_7,401807.277715,1.018416,0.981917,409206.869605,False,False,False
2025-07,dev_5,dev_6,531403.923944,1.063722,0.940095,565266.127051,False,False,False
2025-08,dev_4,dev_5,378209.883552,1.075664,0.929658,406826.914791,False,False,False
2025-09,dev_3,dev_4,518520.051557,1.050208,0.952192,544554.142622,False,False,False


## Ratio Layout

### Step 28 — Build the actual ratio layout

Purpose: Compute all standard diagnostic ratios from the latest cumulative diagonal and format them in the long-form `ratio_frame` layout (one row per period × ratio).  
Uses: `common_actual_ratios`, `latest_cumulative_actual`, `ratio_frame`.  
Produces: `actual_ratio_layout` — long-format DataFrame with columns `period`, `value_type`, `concept`, `concept_value`, `base`, `base_value`, `ratio_name`, `ratio`.

In [28]:
actual_ratios = common_actual_ratios(triangles)

loss_net_actual = (
    latest_cumulative_actual(triangles["Loss Incurred"])
    + latest_cumulative_actual(triangles["ALAE"])
    - latest_cumulative_actual(triangles["Salvages"])
    - latest_cumulative_actual(triangles["Subrogation"])
)
earned_premium_actual = latest_cumulative_actual(triangles["Earned Premium"])
exposure_actual = latest_cumulative_actual(triangles["Exposure"])

actual_ratio_numerators = pd.DataFrame(
    {
        "Loss Incurred Ratio": latest_cumulative_actual(triangles["Loss Incurred"]),
        "Loss Paid Ratio": latest_cumulative_actual(triangles["Loss Paid"]),
        "Loss Net Ratio": loss_net_actual,
        "Claims Reported Frequency": latest_cumulative_actual(triangles["Claims Reported"]),
        "Claims Paid Frequency": latest_cumulative_actual(triangles["Claims Paid"]),
        "Avg Loss Incurred": latest_cumulative_actual(triangles["Loss Incurred"]),
        "Avg Loss Paid": latest_cumulative_actual(triangles["Loss Paid"]),
        "Loss Net Severity": loss_net_actual,
        "ALAE to Loss Incurred": latest_cumulative_actual(triangles["ALAE"]),
        "Salvages to Loss Incurred": latest_cumulative_actual(triangles["Salvages"]),
        "Subrogation to Loss Incurred": latest_cumulative_actual(triangles["Subrogation"]),
        "Average Earned Premium": earned_premium_actual,
    }
)
actual_ratio_denominators = pd.DataFrame(
    {
        "Loss Incurred Ratio": earned_premium_actual,
        "Loss Paid Ratio": earned_premium_actual,
        "Loss Net Ratio": earned_premium_actual,
        "Claims Reported Frequency": exposure_actual,
        "Claims Paid Frequency": exposure_actual,
        "Avg Loss Incurred": latest_cumulative_actual(triangles["Claims Reported"]),
        "Avg Loss Paid": latest_cumulative_actual(triangles["Claims Paid"]),
        "Loss Net Severity": latest_cumulative_actual(triangles["Claims Reported"]),
        "ALAE to Loss Incurred": latest_cumulative_actual(triangles["Loss Incurred"]),
        "Salvages to Loss Incurred": latest_cumulative_actual(triangles["Loss Incurred"]),
        "Subrogation to Loss Incurred": latest_cumulative_actual(triangles["Loss Incurred"]),
        "Average Earned Premium": exposure_actual,
    }
)

actual_ratio_layout = ratio_frame(
    actual_ratios,
    value_type="Actual",
    numerators=actual_ratio_numerators,
    denominators=actual_ratio_denominators,
)

actual_ratio_layout

,period,value_type,concept,concept_value,base,base_value,ratio_name,ratio
0,2016-01,Actual,Loss Incurred,175796.196292,Earned Premium,280199.358984,Loss Incurred Ratio,0.627397
1,2016-02,Actual,Loss Incurred,183746.248505,Earned Premium,292128.983777,Loss Incurred Ratio,0.628990
2,2016-03,Actual,Loss Incurred,184100.501488,Earned Premium,303119.570960,Loss Incurred Ratio,0.607353
3,2016-04,Actual,Loss Incurred,196951.796202,Earned Premium,315137.099414,Loss Incurred Ratio,0.624972
4,2016-05,Actual,Loss Incurred,201165.375549,Earned Premium,326883.913518,Loss Incurred Ratio,0.615403
...,...,...,...,...,...,...,...,...
1435,2025-08,Actual,Earned Premium,720169.657410,Exposure,1457.797308,Average Earned Premium,494.012201
1436,2025-09,Actual,Earned Premium,686734.390031,Exposure,1386.431746,Average Earned Premium,495.325061
1437,2025-10,Actual,Earned Premium,631540.272817,Exposure,1316.360278,Average Earned Premium,479.762481
1438,2025-11,Actual,Earned Premium,606012.565802,Exposure,1247.970280,Average Earned Premium,485.598556


### Step 29 — Define development helper and compute expected ratios for all concepts

Purpose: Define a reusable `development_outputs` helper that runs the standard simple-latest-12 selection, then apply it to Loss Paid, ALAE, Salvages, Subrogation, Claims Reported, and Claims Paid to obtain projected ultimates; then derive expected ratios by dividing each concept's ultimate by its appropriate denominator.  
Uses: `ata_matrix`, `atu_matrix`, `ultimate_matrix`, `calculate_ratio`, `latest_cumulative_actual`.  
Produces: `expected_ratio_layout` — long-format DataFrame analogous to `actual_ratio_layout` with `value_type="Ultimate"`.  

Interpretation: The `development_outputs` helper uses `selected="S - 12"` (simple latest 12) as a uniform illustrative selection for all non-Loss-Incurred concepts. These selections must be calibrated separately before production use.

In [29]:
def development_outputs(cumulative_triangle, *, tail_factor=1.000001):
    ata_result = ata_matrix(
        cumulative_triangle,
        average_specs,
        selected="S - 12",
        tail_factor=tail_factor,
    )
    atu_result = atu_matrix(ata_result)
    ultimate_result = ultimate_matrix(cumulative_triangle, atu_result)
    return ata_result, atu_result, ultimate_result

loss_paid_ata, loss_paid_atu, loss_paid_ultimate = development_outputs(triangles["Loss Paid"])
alae_ata, alae_atu, alae_ultimate = development_outputs(triangles["ALAE"])
salvages_ata, salvages_atu, salvages_ultimate = development_outputs(triangles["Salvages"])
subrogation_ata, subrogation_atu, subrogation_ultimate = development_outputs(triangles["Subrogation"])
claims_reported_ata, claims_reported_atu, claims_reported_ultimate = development_outputs(triangles["Claims Reported"])
claims_paid_ata, claims_paid_atu, claims_paid_ultimate = development_outputs(triangles["Claims Paid"])

earned_premium_actual = latest_cumulative_actual(triangles["Earned Premium"])
exposure_actual = latest_cumulative_actual(triangles["Exposure"])

loss_incurred_ultimate = ultimate.rename(columns={"Ultimate": "Loss Incurred Ultimate"})
loss_paid_ultimate = loss_paid_ultimate.rename(columns={"Ultimate": "Loss Paid Ultimate"})
alae_ultimate = alae_ultimate.rename(columns={"Ultimate": "ALAE Ultimate"})
salvages_ultimate = salvages_ultimate.rename(columns={"Ultimate": "Salvages Ultimate"})
subrogation_ultimate = subrogation_ultimate.rename(columns={"Ultimate": "Subrogation Ultimate"})
claims_reported_ultimate = claims_reported_ultimate.rename(columns={"Ultimate": "Claims Reported Ultimate"})
claims_paid_ultimate = claims_paid_ultimate.rename(columns={"Ultimate": "Claims Paid Ultimate"})

loss_net_ultimate = (
    loss_incurred_ultimate["Loss Incurred Ultimate"]
    + alae_ultimate["ALAE Ultimate"]
    - salvages_ultimate["Salvages Ultimate"]
    - subrogation_ultimate["Subrogation Ultimate"]
)

expected_ratios = pd.DataFrame(
    {
        "Loss Incurred Ratio": calculate_ratio(loss_incurred_ultimate["Loss Incurred Ultimate"], earned_premium_actual),
        "Loss Paid Ratio": calculate_ratio(loss_paid_ultimate["Loss Paid Ultimate"], earned_premium_actual),
        "Loss Net Ratio": calculate_ratio(loss_net_ultimate, earned_premium_actual),
        "Claims Reported Frequency": calculate_ratio(claims_reported_ultimate["Claims Reported Ultimate"], exposure_actual),
        "Claims Paid Frequency": calculate_ratio(claims_paid_ultimate["Claims Paid Ultimate"], exposure_actual),
        "Avg Loss Incurred": calculate_ratio(loss_incurred_ultimate["Loss Incurred Ultimate"], claims_reported_ultimate["Claims Reported Ultimate"]),
        "Avg Loss Paid": calculate_ratio(loss_paid_ultimate["Loss Paid Ultimate"], claims_paid_ultimate["Claims Paid Ultimate"]),
        "Loss Net Severity": calculate_ratio(loss_net_ultimate, claims_reported_ultimate["Claims Reported Ultimate"]),
        "ALAE to Loss Incurred": calculate_ratio(alae_ultimate["ALAE Ultimate"], loss_incurred_ultimate["Loss Incurred Ultimate"]),
        "Salvages to Loss Incurred": calculate_ratio(salvages_ultimate["Salvages Ultimate"], loss_incurred_ultimate["Loss Incurred Ultimate"]),
        "Subrogation to Loss Incurred": calculate_ratio(subrogation_ultimate["Subrogation Ultimate"], loss_incurred_ultimate["Loss Incurred Ultimate"]),
        "Average Earned Premium": calculate_ratio(earned_premium_actual, exposure_actual),
    }
)
expected_ratio_numerators = pd.DataFrame(
    {
        "Loss Incurred Ratio": loss_incurred_ultimate["Loss Incurred Ultimate"],
        "Loss Paid Ratio": loss_paid_ultimate["Loss Paid Ultimate"],
        "Loss Net Ratio": loss_net_ultimate,
        "Claims Reported Frequency": claims_reported_ultimate["Claims Reported Ultimate"],
        "Claims Paid Frequency": claims_paid_ultimate["Claims Paid Ultimate"],
        "Avg Loss Incurred": loss_incurred_ultimate["Loss Incurred Ultimate"],
        "Avg Loss Paid": loss_paid_ultimate["Loss Paid Ultimate"],
        "Loss Net Severity": loss_net_ultimate,
        "ALAE to Loss Incurred": alae_ultimate["ALAE Ultimate"],
        "Salvages to Loss Incurred": salvages_ultimate["Salvages Ultimate"],
        "Subrogation to Loss Incurred": subrogation_ultimate["Subrogation Ultimate"],
        "Average Earned Premium": earned_premium_actual,
    }
)
expected_ratio_denominators = pd.DataFrame(
    {
        "Loss Incurred Ratio": earned_premium_actual,
        "Loss Paid Ratio": earned_premium_actual,
        "Loss Net Ratio": earned_premium_actual,
        "Claims Reported Frequency": exposure_actual,
        "Claims Paid Frequency": exposure_actual,
        "Avg Loss Incurred": claims_reported_ultimate["Claims Reported Ultimate"],
        "Avg Loss Paid": claims_paid_ultimate["Claims Paid Ultimate"],
        "Loss Net Severity": claims_reported_ultimate["Claims Reported Ultimate"],
        "ALAE to Loss Incurred": loss_incurred_ultimate["Loss Incurred Ultimate"],
        "Salvages to Loss Incurred": loss_incurred_ultimate["Loss Incurred Ultimate"],
        "Subrogation to Loss Incurred": loss_incurred_ultimate["Loss Incurred Ultimate"],
        "Average Earned Premium": exposure_actual,
    }
)

expected_ratio_layout = ratio_frame(
    expected_ratios,
    value_type="Ultimate",
    numerators=expected_ratio_numerators,
    denominators=expected_ratio_denominators,
)
expected_ratio_layout

,period,value_type,concept,concept_value,base,base_value,ratio_name,ratio
0,2016-01,Ultimate,Loss Incurred,175796.372088,Earned Premium,280199.358984,Loss Incurred Ratio,0.627397
1,2016-02,Ultimate,Loss Incurred,183746.432251,Earned Premium,292128.983777,Loss Incurred Ratio,0.628991
2,2016-03,Ultimate,Loss Incurred,184100.685588,Earned Premium,303119.570960,Loss Incurred Ratio,0.607353
3,2016-04,Ultimate,Loss Incurred,196951.993154,Earned Premium,315137.099414,Loss Incurred Ratio,0.624972
4,2016-05,Ultimate,Loss Incurred,201165.576714,Earned Premium,326883.913518,Loss Incurred Ratio,0.615404
...,...,...,...,...,...,...,...,...
1435,2025-08,Ultimate,Earned Premium,720169.657410,Exposure,1457.797308,Average Earned Premium,494.012201
1436,2025-09,Ultimate,Earned Premium,686734.390031,Exposure,1386.431746,Average Earned Premium,495.325061
1437,2025-10,Ultimate,Earned Premium,631540.272817,Exposure,1316.360278,Average Earned Premium,479.762481
1438,2025-11,Ultimate,Earned Premium,606012.565802,Exposure,1247.970280,Average Earned Premium,485.598556


### Step 30 — Combine actual and expected ratio layouts

Purpose: Concatenate the actual and ultimate ratio layout DataFrames into a single long-format table for joint display and export.  
Uses: `pd.concat`.  
Produces: `ratio_layout` — unified long-format DataFrame.

In [30]:
ratio_layout = pd.concat(
    [actual_ratio_layout, expected_ratio_layout],
    ignore_index=True,
)

ratio_layout

,period,value_type,concept,concept_value,base,base_value,ratio_name,ratio
0,2016-01,Actual,Loss Incurred,175796.196292,Earned Premium,280199.358984,Loss Incurred Ratio,0.627397
1,2016-02,Actual,Loss Incurred,183746.248505,Earned Premium,292128.983777,Loss Incurred Ratio,0.628990
2,2016-03,Actual,Loss Incurred,184100.501488,Earned Premium,303119.570960,Loss Incurred Ratio,0.607353
3,2016-04,Actual,Loss Incurred,196951.796202,Earned Premium,315137.099414,Loss Incurred Ratio,0.624972
4,2016-05,Actual,Loss Incurred,201165.375549,Earned Premium,326883.913518,Loss Incurred Ratio,0.615403
...,...,...,...,...,...,...,...,...
2875,2025-08,Ultimate,Earned Premium,720169.657410,Exposure,1457.797308,Average Earned Premium,494.012201
2876,2025-09,Ultimate,Earned Premium,686734.390031,Exposure,1386.431746,Average Earned Premium,495.325061
2877,2025-10,Ultimate,Earned Premium,631540.272817,Exposure,1316.360278,Average Earned Premium,479.762481
2878,2025-11,Ultimate,Earned Premium,606012.565802,Exposure,1247.970280,Average Earned Premium,485.598556


### Step 31 — Pivot to concept × value-type table

Purpose: Reshape the long layout into a pivot with `period`, `concept`, `base`, and `ratio_name` as the index and `value_type` (Actual / Ultimate) as columns, for direct Actual-vs-Ultimate comparison.  
Uses: `DataFrame.pivot_table`.  
Produces: `ratio_table`.

In [31]:
ratio_table = ratio_layout.pivot_table(
    index=["period", "concept", "base", "ratio_name"],
    columns="value_type",
    values="ratio",
    aggfunc="first",
)

ratio_table

value_type                                                                  Actual  \
period  concept         base            ratio_name                                   
2016-01 ALAE            Loss Incurred   ALAE to Loss Incurred             0.070420   
        Claims Paid     Exposure        Claims Paid Frequency             0.084010   
        Claims Reported Exposure        Claims Reported Frequency         0.084010   
        Earned Premium  Exposure        Average Earned Premium          307.236139   
        Loss Incurred   Claims Reported Avg Loss Incurred              2294.467262   
...                                                                            ...   
2025-12 Loss Net        Earned Premium  Loss Net Ratio                    0.652863   
        Loss Paid       Claims Paid     Avg Loss Paid                 46401.435485   
                        Earned Premium  Loss Paid Ratio                   0.194019   
        Salvages        Loss Incurred   Salvages to Loss Incurred         0.000000   
        Subrogation     Loss Incurred   Subrogation to Loss Incurred      0.000000   

value_type                                                                Ultimate  
period  concept         base            ratio_name                                  
2016-01 ALAE            Loss Incurred   ALAE to Loss Incurred             0.070420  
        Claims Paid     Exposure        Claims Paid Frequency             0.084010  
        Claims Reported Exposure        Claims Reported Frequency         0.084010  
        Earned Premium  Exposure        Average Earned Premium          307.236139  
        Loss Incurred   Claims Reported Avg Loss Incurred              2294.467262  
...                                                                            ...  
2025-12 Loss Net        Earned Premium  Loss Net Ratio                    0.754633  
        Loss Paid       Claims Paid     Avg Loss Paid                 38356.897057  
                        Earned Premium  Loss Paid Ratio                   0.600287  
        Salvages        Loss Incurred   Salvages to Loss Incurred         0.000000  
        Subrogation     Loss Incurred   Subrogation to Loss Incurred      0.000000  

[1440 rows x 2 columns]

### Step 32 — Pivot to period × (value_type, ratio_name) wide table

Purpose: Reshape the long layout into a wide table with one row per period and multi-level columns (value_type, ratio_name) for export to a spreadsheet-friendly format.  
Uses: `DataFrame.pivot_table`.  
Produces: `ratio_table_wide`.

In [32]:
ratio_table_wide = ratio_layout.pivot_table(
    index="period",
    columns=["value_type", "ratio_name"],
    values="ratio",
    aggfunc="first",
)

ratio_table_wide

value_type                Actual                                           \
ratio_name ALAE to Loss Incurred Average Earned Premium Avg Loss Incurred   
period                                                                      
2016-01                 0.070420             307.236139       2294.467262   
2016-02                 0.076079             310.478248       2323.492296   
2016-03                 0.060524             309.274126       2172.370027   
2016-04                 0.072531             312.016930       2198.544350   
2016-05                 0.072062             311.140218       2233.072141   
...                          ...                    ...               ...   
2025-08                 0.068989             494.012201       6698.703264   
2025-09                 0.068148             495.325061       9802.663734   
2025-10                 0.069721             479.762481      12901.225560   
2025-11                 0.083293             485.598556      19752.766492   
2025-12                 0.075828             481.885154      39254.546127   

value_type                                                                \
ratio_name Avg Loss Paid Claims Paid Frequency Claims Reported Frequency   
period                                                                     
2016-01      2294.467262              0.084010                  0.084010   
2016-02      2323.492296              0.084049                  0.084049   
2016-03      2172.370027              0.086467                  0.086467   
2016-04      2198.544350              0.088696                  0.088696   
2016-05      2233.072141              0.085746                  0.085746   
...                  ...                   ...                       ...   
2025-08      6981.334364              0.021137                  0.038730   
2025-09     10230.502076              0.020564                  0.038152   
2025-10     14249.411754              0.008276                  0.023194   
2025-11     23076.490987              0.003937                  0.013095   
2025-12     46401.435485              0.002015                  0.007450   

value_type                                                       \
ratio_name Loss Incurred Ratio Loss Net Ratio Loss Net Severity   
period                                                            
2016-01               0.627397       0.477863       1747.605290   
2016-02               0.628990       0.481297       1777.913821   
2016-03               0.607353       0.457186       1635.256939   
2016-04               0.624972       0.477652       1680.297416   
2016-05               0.615403       0.470890       1708.688640   
...                        ...            ...               ...   
2025-08               0.525168       0.549613       7010.509612   
2025-09               0.755052       0.789951      10255.756870   
2025-10               0.623701       0.667186      13800.709852   
2025-11               0.532664       0.577031      21398.037052   
2025-12               0.606847       0.652863      42231.130173   

value_type                  ...          Ultimate                \
ratio_name Loss Paid Ratio  ... Avg Loss Incurred Avg Loss Paid   
period                      ...                                   
2016-01           0.627397  ...       2294.467262   2294.467262   
2016-02           0.628990  ...       2323.492296   2323.492296   
2016-03           0.607353  ...       2172.370027   2172.370027   
2016-04           0.624972  ...       2198.544350   2198.544350   
2016-05           0.615403  ...       2233.072141   2233.072141   
...                    ...  ...               ...           ...   
2025-08           0.298706  ...       3029.797691   6444.313253   
2025-09           0.424723  ...       3728.709844   9317.356449   
2025-10           0.245801  ...       3860.537107  12562.439846   
2025-11           0.187075  ...       4072.430728  19542.089346   
2025-12           0.194019  ...       4023.516

## Export Ratio Tables

### Step 33 — Export ratio tables to Excel and CSV

Purpose: Write the Development Method ratio outputs to `data/output/`.  
Produces: `development_method_ratio_layout.xlsx` (3 sheets: Long Layout, Actual vs Ultimate, Wide Table) and `development_method_ratio_layout.csv` (long layout).

In [33]:
output_dir = Path.cwd() / "data" / "output"

if not output_dir.exists():
    output_dir = Path.cwd().parent / "data" / "output"

output_dir.mkdir(parents=True, exist_ok=True)

ratio_layout_path = output_dir / "development_method_ratio_layout.xlsx"
ratio_layout_csv_path = output_dir / "development_method_ratio_layout.csv"

with pd.ExcelWriter(ratio_layout_path) as writer:
    ratio_layout.to_excel(writer, sheet_name="Long Layout", index=False)
    ratio_table.reset_index().to_excel(writer, sheet_name="Actual vs Ultimate", index=False)
    ratio_table_wide.to_excel(writer, sheet_name="Wide Table")

ratio_layout.to_csv(ratio_layout_csv_path, index=False)

ratio_layout_path, ratio_layout_csv_path

(PosixPath('/mnt/data/Linux/Documents/IBNR_Utils/data/output/development_method_ratio_layout.xlsx'),
 PosixPath('/mnt/data/Linux/Documents/IBNR_Utils/data/output/development_method_ratio_layout.csv'))